# 构建图像生成应用程序（OpenAI）

大型语言模型不仅仅用于文本生成。你也可以根据文本描述生成图像。作为一种模态，图像在医疗技术、建筑、旅游、游戏开发、市场营销等领域非常有用。在本课中，我们将使用 OpenAI 平台上的当今<strong>GPT 图像</strong>模型。

## 学习目标

在本课结束时，你将能够：

- 解释什么是图像生成以及它的应用场景。
- 了解`gpt-image`模型家族及其与传统DALL·E模型的区别。
- 构建图像生成应用程序并编辑图像。

## 什么是图像生成？

图像生成模型根据文本提示创建图像。现代模型如`gpt-image`在训练过程中学习文本与图像之间的关系，然后通过迭代将随机噪声转换为与你的提示相符的图像。

两个知名的图像模型家族是：

- **`gpt-image`（OpenAI）** — 本课使用的当前一代模型。支持文本到图像的生成和图像编辑（带掩码的修补）。
- **Midjourney** — 一款流行的第三方模型，拥有自己的服务和基于 Discord 的工作流。

> 旧版OpenAI图像模型——<strong>DALL·E 2</strong>和<strong>DALL·E 3</strong>——属于遗留模型。DALL·E 3 不再支持新部署，且像`create_variation`这样的功能仅存在于DALL·E 2中。新应用请使用`gpt-image`模型。

> **重要提示：** `gpt-image`模型返回生成的图像为<strong>base64</strong>格式（`b64_json`），而非 URL。你的代码需要解码base64字符串为字节并保存——没有可下载的图像URL。


## 构建你的第一个图像生成应用程序

那么构建一个图像生成应用程序需要什么呢？你需要以下库：

- **python-dotenv**，强烈建议使用此库将你的秘密保存在一个与代码分离的 *.env* 文件中。
- **openai**，这个库是你用来与 OpenAI API 交互的工具。
- **pillow**，用于在 Python 中处理图像。


1. 创建一个名为 *.env* 的文件，内容如下：

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. 将上述库收集到一个名为 *requirements.txt* 的文件中，如下所示：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接下来，创建虚拟环境并安装这些库：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 对于 Windows，使用以下命令创建并激活你的虚拟环境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名为 *app.py* 的文件中添加以下代码：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # 创建 OpenAI 对象（从你的 .env 文件读取 OPENAI_API_KEY）
    client = openai.OpenAI()


    try:
        # 使用图像生成 API 创建图像
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此输入你的提示文本
            size='1024x1024',
            n=1
        )
        # 设置存储图像的目录
        image_dir = os.path.join(os.curdir, 'images')

        # 如果目录不存在，则创建它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化图像路径（注意文件类型应为 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型返回的是图像的 base64（b64_json），而不是 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在默认图像查看器中显示图像
        image = Image.open(image_path)
        image.show()

    # 捕获异常
    except openai.BadRequestError as err:
        print(err)

    ```

让我们解释这段代码：

- 首先，我们导入所需的库，包括 OpenAI 库、dotenv 库、base64 模块和 Pillow 库。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- 之后，我们创建客户端，它从你的 ``.env`` 文件读取 API 密钥。

    ```python
    # 创建 OpenAI 对象
    client = openai.OpenAI()
    ```

- 接下来，我们生成图像：

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此输入您的提示文本
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` 模型以 **base64** 字符串形式返回图像，存储于 `data[0].b64_json`。我们将其解码成字节，然后写入文件 —— 没有可下载的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最后，我们打开图像并使用标准的图像查看器显示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 关于生成图像的更多细节

让我们看看 `images.generate` 的参数：

- **model**，用于指定图像模型（例如 `gpt-image-1`）。
- **prompt**，用于生成图像的文本提示。这里是 “Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils”。
- **size**，生成图像的尺寸（例如 `1024x1024`，`1536x1024`，`1024x1536`，或 `"auto"`）。
- **n**，生成的图像数量。这里生成了一张。

> 图像模型不支持 `temperature` 参数 —— 这是文本生成的调控参数。若想获得多样性，请多次调用 API；若想减少多样性，请让提示更具体。

## 图像生成的其他功能

你已看到用几行 Python 代码生成图像。`gpt-image` 模型还可以 <strong>编辑</strong> 已有图像。通过提供图像、可选的 **mask**（标记要更改的区域）和提示，可以修改图像的一部分 —— 例如，给兔子加帽子。

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 编辑内容也作为 base64 返回
edited_image = base64.b64decode(response.data[0].b64_json)
```

基础图像仅含兔子；最终图像中兔子戴上了帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
